# langchain+ollama的例子

## 入门例子

In [10]:
from langchain_ollama.llms import OllamaLLM
from langchain_ollama.llms import OllamaLLM
from langchain import PromptTemplate, LLMChain


model = model = OllamaLLM(model="deepseek-r1:1.5b",
                  base_url="http://192.168.66.17:11434")
model.invoke("Come up with 10 names for a song about parrots")

'<think>\nAlright, the user is asking me to come up with 10 names for a song about parrots. Hmm, I should figure out their intent here. They might be writing a song, maybe for a creative project or just personal interest. Parrots are a common motif in music, especially with themes like flight, mystery, and nature.\n\nI need to think of catchy and memorable names that evoke the imagery of parrots. Words like "flapping," "flame," "flicker" come to mind because they directly relate to parrots\' behaviors. Maybe include some elements of mystery or hidden stories since parrots are often associated with nature\'s surprises.\n\nI should consider the structure of song titles too—sometimes names have a playful, poetic feel. Words like "mystery" or "riddle" could work well if paired with something intriguing about parrots. Also, including words that suggest flight or movement might make it fit for a song title.\n\nLet me brainstorm some combinations: "Flapping Fowl" sounds nice because it combin

In [12]:
from langchain_ollama.llms import OllamaLLM
from langchain import PromptTemplate, LLMChain

# 初始化 Ollama 模型
ollama_llm = OllamaLLM(model="deepseek-r1:1.5b",
                  base_url="http://192.168.66.17:11434")

# 定义提示模板
template = "Translate the following English text to Chinese: {text}"
prompt = PromptTemplate(input_variables=["text"], template=template)

# 创建 LLMChain
llm_chain = LLMChain(prompt=prompt, llm=ollama_llm)

# 输入文本
input_text = "Hello, how are you?"

# 调用 Ollama 模型进行翻译
translated_text = llm_chain.invoke(input_text)

print(translated_text)

{'text': '<think>\nOkay, so I need to translate "Hello, how are you?" from English to Chinese. First, I\'ll break down the sentence into parts.\n\n"Hello," is straightforward in Chinese, it\'s just "你好！" which means "hello!". Then comes the question part: "how are you?". In Chinese, the way to ask someone what they\'re doing or how they\'re feeling is often "都好吗？", meaning "are you fine?" So putting it together with some natural flow, maybe something like:\n\n你好，都好吗？\n\nThat seems simple and direct. I wonder if there\'s a more poetic way to phrase it. Maybe using "你怎么样了？" instead? Let me check.\n\n好，你怎么样了？\n\nHmm, both are good, but the first version is simpler and often used in conversational context. So perhaps the user prefers the simpler sentence.\n\nSo my final answer would be:\n\n你好，都好吗？\n</think>\n\n你好，都好吗？'}


## 知识库例子

rag 的核心流程

先将知识库内容切分
再把切分后的内容向量化存入向量数据库
用户提问之后，先将问题在向量库中进行相似性检索，找出匹配度高的答案。
然后把查询出来的结果，包装好 Prompt。
最后调用大语言模型，让大语言模型基于上一步的结果进行分析并形成最终的答案，返回给用户

In [20]:
import requests

# Step 1: 设置知识库
knowledge_base = {
    "Python": "Python is a high-level programming language.",
    "LangChain": "LangChain is a framework for developing applications with language models.",
    "Ollama": "Ollama is an open-source tool to run large language models locally."
}

# Step 2: 创建获取答案的函数
def get_answer(query):
    """从知识库获取答案。如果找不到，则返回None。"""
    for key in knowledge_base:
        if key.lower() in query.lower():
            return knowledge_base[key]
    return None

# Step 3: 与 Ollama 交互的函数
def ask_ollama(prompt):
    """通过 Ollama API 获取答案。"""
    url = "http://192.168.66.17:11434/ask"  # 修改为你的 Ollama API 地址
    response = requests.post(url, json={"prompt": prompt})
    
    if response.status_code == 200:
        return response.json().get('response')
    else:
        return "Error contacting the model."

# Step 4: 主循环
if __name__ == "__main__":
    print("欢迎使用知识库查询系统！")
    while True:
        user_query = input("请输入你的问题 (或输入 'exit' 来结束): ")
        if user_query.lower() == 'exit':
            print("退出系统。再见！")
            break
        
        # 先从知识库获取答案
        answer = get_answer(user_query)
        
        if answer:
            print("知识库回复: ", answer)
        else:
            # 如果知识库中没有答案，通过 Ollama 获取答案
            ollama_prompt = f"Please provide information about: {user_query}"
            llm_response = ask_ollama(ollama_prompt)
            print("Ollama 回复: ", llm_response)

欢迎使用知识库查询系统！
知识库回复:  Python is a high-level programming language.
知识库回复:  Python is a high-level programming language.
Ollama 回复:  Error contacting the model.
Ollama 回复:  Error contacting the model.
Ollama 回复:  Error contacting the model.
Ollama 回复:  Error contacting the model.
Ollama 回复:  Error contacting the model.
退出系统。再见！


## 本地知识库 RAG 流程（支持 PDF/文本文件加载、向量检索和生成回答）：
1. 文档加载：
   1. 支持 PDF 和 TXT 文件，使用 PyPDFLoader 或 TextLoader。
   2. 更复杂的场景可以添加其他加载器（如 Word/HTML）。
2. 文本分割：
   1. RecursiveCharacterTextSplitter 将长文档分割为小文本块，设置 chunk_size（块大小）和 chunk_overlap（重叠部分）。
3. 嵌入模型：
   1. 使用 OllamaEmbeddings 在本地生成文本嵌入，需与 Ollama 服务中的模型一致（例如 llama2）。
4. 向量数据库：
   1. FAISS.from_documents 自动处理嵌入生成和向量存储。
   2. 使用 save_local 保存数据库，后续可直接加载无需重新计算。
5. 大语言模型：
   1. Ollama() 初始化本地 LLM，确保与嵌入模型使用相同的基础模型。
6. 检索增强生成：
   1. RetrievalQA 结合检索和生成步骤，search_kwargs={"k": 3} 表示检索前 3 个相关文本块。


In [2]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain_community.llms import Ollama

# 步骤 1: 加载本地文档（支持 PDF/TXT）
def load_documents(file_path): 
    if file_path.endswith(".pdf"):
        loader = PyPDFLoader(file_path)
    elif file_path.endswith(".txt"):
        loader = TextLoader(file_path)
    elif file_path.endswith(".md"):
        loader = TextLoader(file_path)
    else:
        raise ValueError("仅支持 PDF 或 TXT 文件")
    return loader.load()

documents = load_documents("/opt/note/xmnote/src/notebook/lua/02_lua数据类型与变量.md")  # 替换为你的文件路径

# 步骤 2: 分割文档为文本块
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
splits = text_splitter.split_documents(documents)

# 步骤 3: 使用 Ollama 生成嵌入（本地运行）
embedding = OllamaEmbeddings(model="deepseek-r1:1.5b",base_url="http://192.168.66.17:11434")  # 与 Ollama 服务中的模型一致

# 步骤 4: 构建向量数据库
vector_db = FAISS.from_documents(
    documents=splits,
    embedding=embedding
)
vector_db.save_local("my_vector_db")  # 保存向量库，无需重复生成

# 步骤 5: 初始化本地大语言模型
llm = Ollama(model="deepseek-r1:1.5b",base_url="http://192.168.66.17:11434")  # 与 embedding 模型相同

# 步骤 6: 构建 RAG 系统
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=vector_db.as_retriever(search_kwargs={"k": 3}),
    chain_type="stuff"  # 简化的处理方式
)

# 步骤 7: 测试查询
question = "文档中提到的主要概念是什么？"  # 替换为你的问题
result = qa_chain({"query": question})
print(f"问题: {question}")
print(f"答案: {result['result']}")

/tmp/ipykernel_84967/3890495470.py:30: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embedding = OllamaEmbeddings(model="deepseek-r1:1.5b",base_url="http://192.168.66.17:11434")  # 与 Ollama 服务中的模型一致


/tmp/ipykernel_84967/3890495470.py:40: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model="deepseek-r1:1.5b",base_url="http://192.168.66.17:11434")  # 与 embedding 模型相同
/tmp/ipykernel_84967/3890495470.py:51: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain({"query": question})


问题: 文档中提到的主要概念是什么？
答案: <think>
嗯，我现在得仔细看看这个问题。首先，文档里提到了很多关于Lua数组和变量的概念，主要是关于数据类型、变量声明、命名规则等等。用户的问题是：“文档中提到的主要概念是什么？”我需要从提供的上下文里找出主要的概念。

让我一个一个看：

文档开头部分讲了有特殊索引方式的数组，说明索引不是只有string或number，还可以是任何类型的值，包括函数和表。然后描述了无key的类型，下标从1开始，并且在表外有t.web和t["web"]这样的表示方法。接着，有数据类型部分，说明总共有8种，前4是基本类型（传值），后4是对象类型（传引用）。变量部分解释了它们的作用域，以及命名规则，特别是变量名必须以字母或下划线开头，并且大写和小写字母敏感。

接着看字符串类型，有两种赋值方式：'hello'或者"helloworld"。然后提到的函数部分，说函数是一种数据类型，是第一类等位元素，可以存储在变量中，可以通过作参数传递给其他函数，也可以作为返回值。

所以，文档中的主要概念应该是 Lua 中的各种基本数据类型、变量、字符串、数组和函数的概念。
</think>

文档中提到的主要概念包括：

1. **Lua 中的特殊索引方式**：数组可以使用任意类型的索引（除了nil），并且下标从1开始。

2. **无key 的类型**：当键为非string或number时，索引的类型由数值决定。表外的键可能表示为 `t.web` 或 `t["web"]`。

3. **数据类型**：总共有8种数据类型，前4种是基本类型（传值），后4种是对象类型（传引用）。

4. **变量声明**：变量可以存储不同类型的值，包括函数和表。变量的命名规则规定了字母、下划线开头，并且大写和小写字母是敏感的。

5. **字符串赋值方式**：字符串可取两种形式，`'hello'` 或 `"hello"`。

6. **函数概念**：函数是一种数据类型，作为第一类等位元素存储，并可以通过参数传递或返回其他函数。


In [5]:
# 构建检索问答链
import os

from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import FAISS

# 实例化Embedding模型
embeddings_model = OllamaEmbeddings(model="deepseek-r1:1.5b",base_url="http://192.168.66.17:11434")  # 与 Ollama 服务中的模型一致

# 加载向量数据库
loaded_db = FAISS.load_local("my_vector_db", embeddings_model, allow_dangerous_deserialization=True)


# 初始化一个 ChatMistralAI 模型实例，并设置温度为 0
llm =  Ollama(model="deepseek-r1:1.5b",base_url="http://192.168.66.17:11434") 

template = """使用以下上下文来回答最后的问题。如果你不知道答案或者不确定结果，只需要你不知道，不要试图编造答
案。最多使用三句话。尽量使答案简明扼要。总是在回答的最后说“谢谢你的提问！要求使用中文回答”。
{context}
问题: {question}
"""
QA_CHAIN_PROMPT = PromptTemplate(input_variables=["context","question"],
                                 template=template)

# 构建检索问答链
qa_chain = RetrievalQA.from_chain_type(llm,
                                       retriever=loaded_db.as_retriever(),
                                       return_source_documents=True,
                                       chain_type_kwargs={"prompt":QA_CHAIN_PROMPT})

# 测试查询
question = "文档中提到的主要概念是什么？"
result = qa_chain({"query": question})
print(f"问题: {question}")
print(f"答案: {result['result']}")

问题: 文档中提到的主要概念是什么？
答案: <think>
嗯，用户的问题是关于Lisp里的'array'类型的。文档里提到了几个概念，我得仔细想想。首先，文档中明确指出这些概念是类似于STL中的unordered_map<string/number, any>的，这让我联想到普通数组在C++中的概念。

然后，文档提到无键类型是用数字作为索引，从1开始，下标依次累加。有键有两种表现形式：一种是表外用t.web，另一种是t["web"]；另一部分则是表内的{web = ...}和[t web ...]这样的结构。这说明数组在不同场合下可以有不同的存储方式。

接着，文档详细讨论了Lisp中的'var'变量的作用域，从声明之后的第一个语句开始，直到包含该声明的最内层语句块结束。这可能涉及到多重嵌套结构中变量的生命周期管理。

函数和表的类型在Lisp里都是可变的，也就是first-class。函数可以作为其他函数的参数传递，也可以返回其他函数，这种特殊能力让我想到类似于Python中的类或对象，可以在不同的地方使用。此外，文档还列举了常见的函数类型，如函数定义、匿名函数等。

最后，在讨论table表时，提到了它是一种基于k-v对的结构，类似于map<k, v>。Lisp中的数组在不同场合下有不同的存储方式和操作方法，这可能涉及到不同的数据结构实现，比如哈希表、散列表等等。

总结一下，主要概念包括普通数组（Lisp中的array）、无键的索引方式（数字从1开始），以及函数和表的特殊类型。需要明确每个概念的具体定义、存储和操作方法。
</think>

在Lisp中，'array'是一种特殊类型的表格，类似于C++中的普通数组，在不同场合下有不同的存储方式和操作方法。

**文档中提到的主要概念：**

1. **普通数组（Lisp's array）**：
   - 无键的类型，索引从1开始，并随着数据的增长而逐步增加。
   - 可用于表外的无键情况（t.web或t["web"]），以及表内的键-值结构。

2. **函数类型（Function type）**：
   - 函数可以存储在变量中，并作为其他函数的参数传递，甚至返回其他函数。类似于C++中的函数定义和匿名函数。

3. **表类型（Table type）**：
   - 基于键-值对的数据结构，用于L

In [ ]:
# 模型微调：使用 Ollama 模型进行微调

LoRA（Low-Rank Adaptation）是一种针对大型语言模型的微调技术，旨在降低微调过程中的计算和内存需求。其核心思想是通过引入低秩矩阵来近似原始模型的全秩矩阵，从而减少参数数量和计算复杂度。

在LoRA中，原始模型的全秩矩阵被分解为低秩矩阵的乘积。具体来说，对于一个全秩矩阵W，LoRA将其分解为两个低秩矩阵A和B的乘积，即W ≈ A * B。其中，A和B的秩远小于W的秩，从而显著减少了参数数量。

图形化微调 ollama-factor